In [1]:
import lenstronomy
from astropy.io import fits
import numpy as np
import scipy as sp
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import minimize
from astropy.wcs import WCS
import pyswarms as ps
from os import listdir
from os.path import isfile, join
from scipy.optimize import linear_sum_assignment
from lenstronomy.Util import param_util
import os, emcee
import copy, math
from tqdm import tqdm
from pathlib import Path
import corner, time
from IPython.display import Image as imp

from vorbin.voronoi_2d_binning import voronoi_2d_binning

from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.Workflow.fitting_sequence import FittingSequence
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.Data.psf import PSF
from lenstronomy.LightModel.Profiles.sersic import Sersic
from lenstronomy.Plots.model_plot import ModelPlot

import sys

sys.path.append("../src")


from iful.util import *
from iful.image_set import *
from iful.flat_modeling import *
from iful.iful_modeling import *

In [ ]:
while not os.path.exists("s4_models/flat_chains.pickle"):
    time.sleep(60)

In [ ]:
with open("s4_models/imset4.pickle", "rb") as handle:
    imset4 = pickle.load(handle)

with open("s4_models/flatmodel4.pickle", "rb") as handle:
    flatmodel4 = pickle.load(handle)

iful_pso_results_filename = "s4_models/iful_pso_results_params.pickle"
if os.path.isfile(iful_pso_results_filename):
    with open(iful_pso_results_filename, "rb") as handle:
        previous_results = pickle.load(handle)
else:
    previous_results = {}

## Setting up ifulmodel object

In [ ]:
# import numpy as np
# import scipy as sp
# import copy
# import matplotlib.pyplot as plt
# from scipy.stats import norm
# from scipy.optimize import minimize
# import lenstronomy.Util.class_creator as class_creator
# from lenstronomy.Util import param_util
# from vorbin.voronoi_2d_binning import voronoi_2d_binning
# from tqdm import tqdm

# # from IFUL.util import *
# # from IFUL.image_set import *
# # from IFUL.flat_modeling import *

# class IFULModel():
#     def __init__(self, imageset, flatmodel, iful_profiles, sourceplane_size, num_bins, num_rsersics, spectral_res, constant_val=0.):
#         self.imset = imageset
#         self.sourceplane_size = sourceplane_size
#         # self.num_bins = num_bins
#         self.num_bins = 0.
#         self.num_rsersics = num_rsersics
#         self.init_fitting_seq = flatmodel.init_pso_fitting_seq
#         self.spectral_res = spectral_res
#         self.constant_val = constant_val
#         self.iful_profiles = iful_profiles

#         if "SERSIC" not in iful_profiles:
#             self.init_fitting_seq.fit_sequence([['update_settings', {'source_add_fixed': [[0, ['n_sersic']]]}], ])

#         self.imModel_classcreator = class_creator.create_im_sim(
#             self.init_fitting_seq.multi_band_list,
#             'multi-linear',
#             self.init_fitting_seq._updateManager.kwargs_model,
#             bands_compute=None,
#             linear_solver=False,
#             image_likelihood_mask_list=np.array([imageset.mask]))
#         kwargs_params = copy.deepcopy(flatmodel.init_pso_fit)
#         kwargs_params.pop("kwargs_tracer_source", None)

#         self.immodel_init = copy.deepcopy(self.imModel_classcreator)
#         self.immodel_init.image_linear_solve(inv_bool=True, **kwargs_params)

#         # immodel_init = immodel_init._imageModel_list[0]
#         self.sm_init = self.immodel_init._imageModel_list[0].source_mapping

#         self.get_sourceplane_img(flatmodel)

#         if np.sum(["VORONOI" in s for s in self.iful_profiles]) >= 1:
#             self.voronoi_given_nbins(num_bins,
#                                      np.nanmax(self.source_fluxes)*2,
#                                      np.nansum(self.source_fluxes) / np.sum(~np.isnan(self.source_fluxes))**0.5 / 2,
#                                      flatmodel.init_pso_fit['kwargs_source'])

#         self.init_sersic_amp = flatmodel.init_pso_fit['kwargs_source'][0]['amp']

#         self.len_model_numparams = self.init_fitting_seq.param_class.num_param()[0]
#         self.v_los_fnc, self.v_los_numparams = self.decide_profiles_fnc(iful_profiles[0], self.num_bins)
#         self.v_disp_fnc, self.v_disp_numparams = self.decide_profiles_fnc(iful_profiles[1], self.num_bins)
#         self.flx_fnc, self.flx_numparams = self.decide_profiles_fnc(iful_profiles[2], self.num_bins)

#         self.obs_datacube = np.transpose(self.imset.datacube, (2, 0, 1))
#         self.var_datacube = np.transpose(self.imset.var_datacube, (2, 0, 1))
#         self.datacube_mask = np.transpose(self.imset.mask_3d, (2, 0, 1))
#         self.datacube_unc = self.imset.brms_3d
#         self.central_wave = np.mean(self.imset.wavelength)

#         self.init_lenstronomy_args = self.init_fitting_seq.param_class.kwargs2args(**flatmodel.init_pso_fit)

#         ra_grid, dec_grid = self.immodel_init._imageModel_list[0].ImageNumerics.coordinates_evaluate
#         self.init_x_source_vals, self.init_y_source_vals = self.sm_init._lens_model.ray_shooting(ra_grid, dec_grid, kwargs_params['kwargs_lens'])

#     def get_num_free_params(self):
#         return self.len_model_numparams + self.v_los_numparams + self.v_disp_numparams + self.flx_numparams

#     def get_sourceplane_img(self, flatmodel):
#         self.sourcecenter = np.array([flatmodel.init_pso_fit['kwargs_source'][0]['center_x'], flatmodel.init_pso_fit['kwargs_source'][0]['center_y']])
#         center_pixel = self.sourceplane_size//2
#         dict_sersic = copy.deepcopy(flatmodel.init_pso_fit['kwargs_source'][0])

#         dpix_mult = 1.
#         not_valid = True

#         while not_valid:
#             dpix = dict_sersic['R_sersic']/self.sourceplane_size*2 * self.num_rsersics * dpix_mult

#             pixel_locations = []
#             values = []

#             teste1 = dict_sersic['e1']
#             teste2 = dict_sersic['e2']

#             source_img = np.zeros((self.sourceplane_size, self.sourceplane_size))
#             for x in np.arange(self.sourceplane_size):
#                 for y in np.arange(self.sourceplane_size):
#                     centered_x, centered_y = (x-center_pixel)*dpix, (y-center_pixel)*dpix

#                     xval = self.sourcecenter[0] + centered_x
#                     yval = self.sourcecenter[1] + centered_y

#                     x_, y_ = param_util.transform_e1e2_product_average(
#                         centered_x, centered_y, teste1, teste2, center_x=0, center_y=0)
#                     d = (x_**2 + y_**2)**0.5

#                     v = self.sm_init._light_model.surface_brightness(xval, yval, [dict_sersic])

#                     if d >= dict_sersic['R_sersic']*self.num_rsersics:
#                         source_img[x, y] = np.nan
#                         continue

#                     pixel_locations += [[x, y]]
#                     values += [v]

#                     source_img[x, y] = v

#             pixel_locations = np.array(pixel_locations)
#             values = np.array(values)

#             not_valid = not is_border_all_nan(source_img)
#             if not_valid:
#                 dpix_mult += 0.01

#         self.pixel_locations = pixel_locations
#         self.source_fluxes = values
#         self.dpix = dpix

#     def voronoi_given_nbins(self, target_y, low, high, kwargs_source, epsilon=1e-7):
#         while (high - low) > epsilon:
#             mid = low + (high - low) / 2.0
#             bin_number, y_gen, x_gen, y_bar, x_bar, sn, nPixels, scale = voronoi_2d_binning(self.pixel_locations.T[1], self.pixel_locations.T[0], self.source_fluxes,
#                                                                                             np.ones(self.source_fluxes.shape), mid, pixelsize=None, plot=False, quiet=True)
#             mid_y = len(y_gen)
#             if mid_y == target_y:
#                 break
#             elif mid_y < target_y:
#                 high = mid
#             else:
#                 low = mid

#         bin_number, y_gen, x_gen, y_bar, x_bar, sn, nPixels, scale = voronoi_2d_binning(self.pixel_locations.T[1], self.pixel_locations.T[0], self.source_fluxes,
#                                                                                         np.ones(self.source_fluxes.shape), mid, pixelsize=None, plot=True, quiet=True)

#         self.init_bin_sourceflux = self.sm_init._light_model.surface_brightness(self.sourcecenter[0] + (x_bar - self.sourceplane_size//2)*self.dpix,
#                                                                            self.sourcecenter[1] + (y_bar - self.sourceplane_size//2)*self.dpix,
#                                                                            kwargs_source)

#         x_, y_ = param_util.transform_e1e2_product_average((x_gen - self.sourceplane_size//2)*self.dpix,
#                                                            (y_gen - self.sourceplane_size//2)*self.dpix,
#                                                            kwargs_source[0]['e1'],
#                                                            kwargs_source[0]['e2'],
#                                                            center_x=0, center_y=0)
#         points_rot = rotate_points(np.array([x_, y_]).T, kwargs_source[0]['e1'], kwargs_source[0]['e2'])
#         x_, y_ = points_rot[:, 0], points_rot[:, 1]

#         self.x_bins = x_ / kwargs_source[0]['R_sersic']
#         self.y_bins = y_ / kwargs_source[0]['R_sersic']
#         self.num_bins = len(self.y_bins)

#     def given_ra_dec_return_bin_no(self, x_source, y_source, source_params, return_dist=False):
#         x_ra, y_dec = param_util.transform_e1e2_product_average(
#             x_source - source_params['center_x'], y_source - source_params['center_y'], source_params['e1'], source_params['e2'], center_x=0, center_y=0
#         )
#         points_rot = rotate_points(np.array([x_ra, y_dec]).T, source_params['e1'], source_params['e2'])
#         x_ra, y_dec = points_rot[:, 0], points_rot[:, 1]

#         x_ra, y_dec = (x_ra, y_dec) / source_params['R_sersic']

#         dists = (x_ra**2 + y_dec**2)**0.5 * source_params['R_sersic']
#         res = find_closest_point_indices(np.array([self.x_bins, self.y_bins]).T, np.array([x_ra, y_dec]).T, self.num_rsersics)

#         if return_dist:
#             return res, dists
#         return res

#     def generate_residuals(self, all_fitted_params, return_datacube=False):
#         assert self.get_num_free_params() == len(all_fitted_params)

#         lens_model_params = all_fitted_params[:self.len_model_numparams]
#         v_los_params = all_fitted_params[self.len_model_numparams : self.len_model_numparams + self.v_los_numparams]
#         v_disp_params = all_fitted_params[-1*(self.flx_numparams + self.v_disp_numparams) : -1*self.flx_numparams]
#         flx_params = all_fitted_params[-1*self.flx_numparams:]

#         kwargs_lenstronomy = self.init_fitting_seq.param_class.args2kwargs(lens_model_params)
#         kwargs_lenstronomy.pop("kwargs_tracer_source", None)

#         if np.any((np.array(self.init_lenstronomy_args) - np.array(lens_model_params))**2 > 1e-5):
#             immodel = copy.deepcopy(self.imModel_classcreator)
#             immodel.image_linear_solve(inv_bool=True, **kwargs_lenstronomy)
#             immodel = immodel._imageModel_list[0]

#             sm = immodel.source_mapping
#             ra_grid, dec_grid = immodel.ImageNumerics.coordinates_evaluate
#             x_source_vals, y_source_vals = sm._lens_model.ray_shooting(ra_grid, dec_grid, kwargs_lenstronomy['kwargs_lens'])

#         else:
#             # print('default detected')
#             immodel = self.immodel_init._imageModel_list[0]
#             x_source_vals, y_source_vals = self.init_x_source_vals, self.init_y_source_vals
#             sm = self.sm_init

#         if self.num_bins > 0:
#             binno = self.given_ra_dec_return_bin_no(x_source_vals, y_source_vals, kwargs_lenstronomy['kwargs_source'][0])
#         else:
#             binno = np.ones(x_source_vals.shape)
#         aux_params = [kwargs_lenstronomy['kwargs_source'], sm, self.constant_val]

#         c = 299792
#         z_los = self.v_los_fnc(x_source_vals, y_source_vals, binno, aux_params, v_los_params) / c
#         v_disp = self.v_disp_fnc(x_source_vals, y_source_vals, binno, aux_params, v_disp_params)
#         flxs = self.flx_fnc(x_source_vals, y_source_vals, binno, aux_params, flx_params)

#         sigma_model = v_disp * self.central_wave / c
#         sigma_total = (sigma_model**2 + (self.central_wave/(2.355 * self.spectral_res))**2)**0.5

#         source_light = np.array([np.zeros(self.imset.wavelength.shape) if np.sum(np.isnan([z, sigma_ang, flx])) > 0 else self.imset.gen_2d_spec_fixratios([z, sigma_ang, flx])
#                                  for z, sigma_ang, flx in zip(z_los, sigma_total, flxs)])

#         model_datacube = []
#         for ii in np.arange(source_light.shape[1]):
#             model_datacube += [immodel.ImageNumerics.re_size_convolve(source_light[:, ii], unconvolved=False)]
#         model_datacube = np.array(model_datacube)

#         res = np.nansum(((model_datacube - self.obs_datacube)**2/self.var_datacube)*self.datacube_mask)

#         if return_datacube:
#             return res, model_datacube
#         return res

#     def generate_source_plots(self, all_fitted_params, image_size=None, dpix=None):
#         assert self.get_num_free_params() == len(all_fitted_params)

#         if image_size is None:
#             image_size = self.sourceplane_size
#         if dpix is None:
#             dpix = self.dpix

#         lens_model_params = all_fitted_params[:self.len_model_numparams]
#         v_los_params = all_fitted_params[self.len_model_numparams : self.len_model_numparams + self.v_los_numparams]
#         v_disp_params = all_fitted_params[-1*(self.flx_numparams + self.v_disp_numparams) : -1*self.flx_numparams]
#         flx_params = all_fitted_params[-1*self.flx_numparams:]

#         kwargs_lenstronomy = self.init_fitting_seq.param_class.args2kwargs(lens_model_params)
#         kwargs_lenstronomy.pop("kwargs_tracer_source", None)

#         if np.any((np.array(self.init_lenstronomy_args) - np.array(lens_model_params))**2 > 1e-5):
#             immodel = copy.deepcopy(self.imModel_classcreator)
#             immodel.image_linear_solve(inv_bool=True, **kwargs_lenstronomy)
#             immodel = immodel._imageModel_list[0]
#             sm = immodel.source_mapping
#         else:
#             immodel = self.immodel_init._imageModel_list[0]
#             sm = self.sm_init

#         c = 299792
#         delta_coor = (np.arange(image_size) - image_size/2) * dpix

#         v_los_img = np.zeros((image_size, image_size))
#         v_disp_img = np.zeros((image_size, image_size))
#         flxs_img = np.zeros((image_size, image_size))
#         for ix, x in enumerate(kwargs_lenstronomy['kwargs_source'][0]['center_x'] + delta_coor):
#             for iy, y in enumerate(kwargs_lenstronomy['kwargs_source'][0]['center_y'] + delta_coor):

#                 if self.num_bins > 0:
#                     binno = self.given_ra_dec_return_bin_no(np.array([x]), np.array([y]), kwargs_lenstronomy['kwargs_source'][0])
#                 else:
#                     binno = 1
#                 aux_params = [kwargs_lenstronomy['kwargs_source'], sm, self.constant_val]

#                 v_los = self.v_los_fnc(np.array([x]), np.array([y]), binno, aux_params, v_los_params)[0]
#                 v_disp = self.v_disp_fnc(np.array([x]), np.array([y]), binno, aux_params, v_disp_params)[0]
#                 flxs = self.flx_fnc(np.array([x]), np.array([y]), binno, aux_params, flx_params)[0]

#                 if np.sum(np.isnan([v_los, v_disp, flxs])) > 0 or np.isnan(binno):
#                     v_los_img[ix, iy] = np.nan
#                     v_disp_img[ix, iy] = np.nan
#                     flxs_img[ix, iy] = np.nan
#                 else:
#                     v_los_img[ix, iy] = v_los
#                     v_disp_img[ix, iy] = v_disp
#                     flxs_img[ix, iy] = flxs
#         v_los_img -= np.nanmedian(v_los_img)

#         fig, axs = plt.subplots(1, 3, figsize=(18, 5))

#         col = axs[0].imshow(v_los_img, cmap="bwr", )
#         fig.colorbar(col, ax=axs[0], label='LOS Velocity [km/s]')

#         col = axs[1].imshow(v_disp_img, )
#         fig.colorbar(col, ax=axs[1], label='Velocity dispersion [km/s]')

#         col = axs[2].imshow(np.log10(flxs_img))
#         fig.colorbar(col, ax=axs[2], label='log10 flux')
#         plt.show()

#     def decide_profiles_fnc(self, profile_name, num_bins):
#         if profile_name == "VORONOI":
#             return self.get_voronoi_v_given_xy_bin, num_bins
#         elif profile_name == "ARCTAN":
#             return self.get_arctan_v_given_xy_bin, 4
#         elif profile_name == "SERSIC":
#             return self.get_sersic_v_given_xy_bin, 1
#         elif profile_name == "GAUSSIAN":
#             return self.get_gaussian_v_given_xy_bin, 2
#         elif profile_name == "POWER_LAW":
#             return self.get_power_law_v_given_xy_bin, 2
#         elif profile_name == "CONSTANT_FIXED":
#             return self.get_constant_v_given_xy_bin, 0
#         elif profile_name == "CONSTANT_FITTED":
#             return self.get_constant_v_given_xy_bin, 1
#         raise Exception("Profile not implemented")

#     @staticmethod
#     def get_voronoi_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
#         # aux_params: [kwargs_source, sm, constant_val]
#         # fitted_params: list same len of num of bins
#         if not check_list(x):
#             if np.isnan(binno):
#                 return 0.
#             return fitted_params[int(binno)]
#         else:
#             return np.array([0. if np.isnan(b) else fitted_params[int(b)] for b in binno])

#     @staticmethod
#     def get_arctan_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
#         # aux_params: [kwargs_source, sm, constant_val]
#         # fitted_params: [v_pa, v_a, v_b, v_c]
#         kwargs_source = aux_params[0]
#         v_pa, v_a, v_b, v_c = fitted_params
#         c_x, c_y = kwargs_source[0]['center_x'], kwargs_source[0]['center_y']

#         if not check_list(x):
#             return arctan_2d(v_pa, v_a, v_b, v_c, c_x, c_y, x, y)
#         else:
#             return np.array([arctan_2d(v_pa, v_a, v_b, v_c, c_x, c_y, xp, yp) for xp, yp in zip(x, y)])

#     @staticmethod
#     def get_sersic_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
#         # aux_params: [kwargs_source, sm, constant_val]
#         # fitted_params: [scale]
#         kwargs_source, sm = aux_params[0], aux_params[1]
#         scale = fitted_params[0]

#         if not check_list(x):
#             return sm._light_model.surface_brightness(np.array([x]), np.array([y]), kwargs_source)[0] * scale
#         else:
#             return sm._light_model.surface_brightness(x, y, kwargs_source) * scale

#     @staticmethod
#     def get_gaussian_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
#         # aux_params: [kwargs_source, sm, constant_val]
#         # fitted_params: [amp, sigma_model]
#         if not check_list(x):
#             kwargs_source = aux_params[0]
#             amp, sigma_model = fitted_params

#             c_x, c_y = kwargs_source[0]['center_x'], kwargs_source[0]['center_y']
#             dist = ((x-c_x)**2 + (y-c_y)**2)**0.5

#             return norm_dist(amp, 0, sigma_intrinsic, dist)
#         else:
#             return np.array([get_gaussian_v_given_xy_bin(x0, y0, binno0, aux_params, fitted_params) for x0, y0, binno0 in zip(x, y, binno)])

#     @staticmethod
#     def get_power_law_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
#         # aux_params: [kwargs_source, sm, constant_val]
#         # fitted_params: [scale, gamma]

#         if not check_list(x):
#             return get_power_law_v_given_xy_bin(np.array([x]), np.array([y]), binno, aux_params, fitted_params)[0]
#         else:
#             kwargs_source = aux_params[0]
#             scale, gamma = fitted_params

#             x_, y_ = param_util.transform_e1e2_product_average(x - kwargs_source[0]['center_x'], y - kwargs_source[0]['center_y'],
#                                                                kwargs_source[0]['e1'], kwargs_source[0]['e2'], center_x=0, center_y=0)
#             dist = (x_**2 + y_**2)**0.5
#             return scale * dist**((2-gamma)/2)

#     @staticmethod
#     def get_constant_v_given_xy_bin(x, y, binno, aux_params, fitted_params=[]):
#         # aux_params: [kwargs_source, sm, sigma_intrinsic, constant_val] or
#         # fitted_params: [constant_val]
#         if len(fitted_params) == 0:
#             const_val = aux_params[-1]
#         else:
#             const_val = fitted_params[0]

#         if not check_list(x):
#             return const_val
#         else:
#             return np.ones(len(x))*const_val

In [ ]:
c = 299792
threadCount = 60

In [ ]:
# imageset, flatmodel, iful_profiles, sourceplane_size, num_bins, num_rsersics, spectral_res, constant_val=0.):
iful_profiles = ["ARCTAN", "POWER_LAW", "SERSIC"]
ifulmodel4 = IFULModel(
    imset4,
    flatmodel4,
    iful_profiles,
    sourceplane_size=100,
    num_bins=100,
    num_rsersics=2,
    spectral_res=3500,
)

## Setting initial parameters

In [ ]:
lens_model_initparams = list(
    ifulmodel4.init_fitting_seq.param_class.kwargs2args(**flatmodel4.init_pso_fit)
)
# v_pa, v_a, v_b, v_c
v_los_initparams = [75, 150, 7, imset4.zs * c]
# scale, gamma
v_disp_initparams = [75, 2.1]
# bin_vals
flx_initparams = [
    ifulmodel4.init_sersic_amp * 25**0.5
]  # list(ifulmodel4.init_bin_sourceflux*25**0.5)

init_params = (
    lens_model_initparams + v_los_initparams + v_disp_initparams + flx_initparams
)
print(len(init_params))

In [ ]:
res, model_datacube = ifulmodel4.generate_residuals(init_params, return_datacube=True)

In [ ]:
ifulmodel4.generate_source_plots(init_params)

In [ ]:
filename = f"s4_models/animation_init.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.datacube_unc**2,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename,
)
imp(filename=filename)

## Fitting only IFUL parameters (exclude lens model parameters)

In [ ]:
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
nwalkers = 240

def min_fnc(set_params):
    all_res = []
    for params in set_params:
        all_res += [ifulmodel4.generate_residuals(lens_model_initparams + list(params))]
    return all_res

iful_lowerbounds = np.array([0, 0, 0, 1.430 * c, 1, 1.0, 0])
iful_upperbounds = np.array([360, 1000, 10, 1.436 * c, 500, 3.0, 1e5])

all_init_fits = [v_los_initparams + v_disp_initparams + flx_initparams]
while len(all_init_fits) < nwalkers:
    all_init_fits_t = [
        np.random.uniform(80, 100),
        np.random.uniform(100, 400),
        np.random.uniform(3, 7),
        np.random.uniform(1.432, 1.434) * c,
        np.random.uniform(50, 150),
        np.random.uniform(1.5, 2.5),
        np.random.uniform(0.5, 3) * ifulmodel4.init_sersic_amp * 25**0.5,
    ]
    all_init_fits_t = np.array(all_init_fits_t)
    if np.all(all_init_fits_t >= iful_lowerbounds) and np.all(
        all_init_fits_t <= iful_upperbounds
    ):
        all_init_fits += [all_init_fits_t]
all_init_fits = np.array(all_init_fits)

optimizer = ps.single.GlobalBestPSO(
    n_particles=nwalkers,
    dimensions=len(all_init_fits[0]),
    options=options,
    bounds=[iful_lowerbounds, iful_upperbounds],
    init_pos=all_init_fits,
)

res_key = "_".join(iful_profiles)+"_ifulonly"
if res_key in previous_results:
    init_params = previous_results[res_key]
else:
    cost, pos = optimizer.optimize(min_fnc, iters=100, n_processes=threadCount)
    init_params = lens_model_initparams + list(pos)
    previous_results[res_key] = init_params
    with open(iful_pso_results_filename, "wb") as handle:
        pickle.dump(previous_results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
def check_bounds_proximity(params, lower_bounds, upper_bounds, atol=1e-3, rtol=1e-5):
    params = np.asarray(params)
    lower_bounds = np.asarray(lower_bounds)
    upper_bounds = np.asarray(upper_bounds)
    lower_margin = lower_bounds + atol + rtol * np.abs(lower_bounds)
    upper_margin = upper_bounds - (atol + rtol * np.abs(upper_bounds))
    at_or_past_lower = params <= lower_margin
    at_or_past_upper = params >= upper_margin
    any_violation = at_or_past_lower | at_or_past_upper

    return {
        "lower_violation": at_or_past_lower,
        "upper_violation": at_or_past_upper,
        "any_violation": any_violation,
        "is_safe": not any_violation.any() 
    }
    
results = check_bounds_proximity(init_params[len(lens_model_initparams):], iful_lowerbounds, iful_upperbounds)
print("Violates Lower Bounds:", results['lower_violation'])
print("Violates Upper Bounds:", results['upper_violation'])
print("Any Violation Map:", results['any_violation'])

In [ ]:
res, model_datacube = ifulmodel4.generate_residuals(init_params, return_datacube=True)

In [ ]:
ifulmodel4.generate_source_plots(init_params)

In [ ]:
filename = f"s4_models/animation_ifulonly.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.datacube_unc**2,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename,
)
imp(filename=filename)

## Fitting all parameters (lensing and IFUL)

In [ ]:
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}

def min_fnc(set_params):
    all_res = []
    for params in set_params:
        all_res += [ifulmodel4.generate_residuals(params)]
    return all_res
    
lensing_lower_bounds, lensing_upper_bounds = ifulmodel4.init_fitting_seq.likelihoodModule.param_limits
iful_lowerbounds = np.array(list(lensing_lower_bounds) + [0, 0, 0, 1.430 * c, 1, 1.0, 0])
iful_upperbounds = np.array(list(lensing_upper_bounds) + [360, 1000, 10, 1.436 * c, 500, 3.0, 1e5])

all_init_fits = [init_params]
while len(all_init_fits) < nwalkers:
    all_init_fits_t = list(np.array(init_params) * np.random.normal(1, 0.01, len(init_params)))
    # all_init_fits_t += [
    #     np.random.uniform(80, 100),
    #     np.random.uniform(100, 400),
    #     np.random.uniform(3, 7),
    #     np.random.uniform(1.432, 1.434) * c,
    #     np.random.uniform(50, 150),
    #     np.random.uniform(1.5, 2.5),
    #     np.random.uniform(0.5, 3) * ifulmodel4.init_sersic_amp * 25**0.5,
    # ]
    all_init_fits_t = np.array(all_init_fits_t)
    if np.all(all_init_fits_t >= iful_lowerbounds) and np.all(
        all_init_fits_t <= iful_upperbounds
    ):
        all_init_fits += [all_init_fits_t]
all_init_fits = np.array(all_init_fits)

optimizer = ps.single.GlobalBestPSO(
    n_particles=nwalkers,
    dimensions=len(all_init_fits[0]),
    options=options,
    bounds=[iful_lowerbounds, iful_upperbounds],
    init_pos=all_init_fits,
)

res_key = "_".join(iful_profiles)+"_ifulall"
if res_key in previous_results:
    init_params = previous_results[res_key]
else:
    cost, pos = optimizer.optimize(min_fnc, iters=500, n_processes=threadCount)
    init_params = list(pos)
    previous_results[res_key] = init_params
    with open(iful_pso_results_filename, "wb") as handle:
        pickle.dump(previous_results, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
results = check_bounds_proximity(pos, iful_lowerbounds, iful_upperbounds)
print("Violates Lower Bounds:", results['lower_violation'])
print("Violates Upper Bounds:", results['upper_violation'])
print("Any Violation Map:", results['any_violation'])

In [ ]:
init_params

In [ ]:
res, model_datacube = ifulmodel4.generate_residuals(init_params, return_datacube=True)

In [ ]:
ifulmodel4.generate_source_plots(init_params)

In [ ]:
filename = f"s4_models/animation_ifulall.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.datacube_unc**2,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename,
)
imp(filename=filename)

In [ ]:
# imageset, flatmodel, iful_profiles, sourceplane_size, num_bins, num_rsersics, spectral_res, constant_val=0.):
iful_profiles = ["ARCTAN", "POWER_LAW", "VORONOI"]
ifulmodel4 = IFULModel(
    imset4,
    flatmodel4,
    iful_profiles,
    sourceplane_size=100,
    num_bins=100,
    num_rsersics=2,
    spectral_res=3500,
)

In [ ]:
lens_model_initparams = list(
    ifulmodel4.init_fitting_seq.param_class.kwargs2args(**flatmodel4.init_pso_fit)
)
# v_pa, v_a, v_b, v_c
v_los_initparams = [75, 150, 7, imset4.zs * c]
# scale, gamma
v_disp_initparams = [75, 2.1]
# bin_vals
flx_initparams = list(ifulmodel4.init_bin_sourceflux * 25**0.5)

init_params = (
    lens_model_initparams + v_los_initparams + v_disp_initparams + flx_initparams
)
print(len(init_params))

In [ ]:
res, model_datacube = ifulmodel4.generate_residuals(init_params, return_datacube=True)

In [ ]:
ifulmodel4.generate_source_plots(init_params)

In [ ]:
filename = f"datacubes/animation_init.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.var_datacube,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename,
)
imp(filename=filename)

In [ ]:
options = {"c1": 0.5, "c2": 0.3, "w": 0.9}
nwalkers = 240


def min_fnc(set_params):
    all_res = []
    for params in set_params:
        all_res += [ifulmodel4.generate_residuals(lens_model_initparams + list(params))]
    return all_res


iful_lowerbounds = np.array(
    [0, 0, 0, 1.430 * c, 1, 1.0] + list(np.zeros(ifulmodel4.init_bin_sourceflux.shape))
)
iful_upperbounds = np.array(
    [360, 1000, 10, 1.436 * c, 500, 3.0]
    + list(
        np.ones(ifulmodel4.init_bin_sourceflux.shape)
        * np.max(ifulmodel4.init_bin_sourceflux.shape)
        * 20
    )
)

all_init_fits = [v_los_initparams + v_disp_initparams + flx_initparams]
while len(all_init_fits) < nwalkers:
    all_init_fits_t = [
        np.random.uniform(80, 100),
        np.random.uniform(100, 400),
        np.random.uniform(3, 7),
        np.random.uniform(1.432, 1.434) * c,
        np.random.uniform(50, 150),
        np.random.uniform(1.5, 2.5),
    ] + list(np.array(flx_initparams) * np.random.normal(1, 0.2, len(flx_initparams)))
    all_init_fits_t = np.array(all_init_fits_t)
    if np.all(all_init_fits_t >= iful_lowerbounds) and np.all(
        all_init_fits_t <= iful_upperbounds
    ):
        all_init_fits += [all_init_fits_t]
all_init_fits = np.array(all_init_fits)

optimizer = ps.single.GlobalBestPSO(
    n_particles=nwalkers,
    dimensions=len(all_init_fits[0]),
    options=options,
    bounds=[iful_lowerbounds, iful_upperbounds],
    init_pos=all_init_fits,
)
cost, pos = optimizer.optimize(min_fnc, iters=300, n_processes=threadCount)

In [ ]:
pos

In [ ]:
lens_model_initparams = list(
    ifulmodel4.init_fitting_seq.param_class.kwargs2args(**flatmodel4.init_pso_fit)
)
init_params = lens_model_initparams + list(pos)
print(len(init_params))

In [ ]:
res, model_datacube = ifulmodel4.generate_residuals(init_params, return_datacube=True)

In [ ]:
ifulmodel4.generate_source_plots(init_params)

In [ ]:
filename = f"datacubes/animation_init.gif"
gen_gif(
    ifulmodel4.obs_datacube,
    model_datacube,
    ifulmodel4.var_datacube,
    ifulmodel4.datacube_mask,
    ifulmodel4.imset.wavelength,
    filename,
)
imp(filename=filename)

In [ ]:
cropped_hdul = fits.HDUList(
    [fits.PrimaryHDU(data=model_datacube, header=flatmodel4.header_wcs)]
)
cropped_hdul.writeto("datacubes/test_datacube.fits", overwrite=True)

In [ ]:
ifulmodel4.len_model_numparams, ifulmodel4.v_los_numparams, ifulmodel4.v_disp_numparams, ifulmodel4.flx_numparams

In [ ]:
binned_lensed_img = ifulmodel4.gen_binned_lensed_whitelight(
    flatmodel4.init_pso_fit["kwargs_source"], flatmodel4.init_pso_fit["kwargs_lens"]
)
plt.imshow(binned_lensed_img)
plt.show()

In [ ]:
binned_unlensed_img = ifulmodel4.gen_binned_unlensed_whitelight(
    flatmodel4.init_pso_fit["kwargs_source"]
)
plt.imshow(binned_unlensed_img)
plt.show()

In [ ]:
def get_voronoi_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
    # aux_params: []
    # fitted_params: list same len of num of bins
    if not check_list(x):
        return fitted_params[int(binno)]
    else:
        return np.array([fitted_params[int(b)] for b in binno])


def get_arctan_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
    # aux_params: [kwargs_source]
    # fitted_params: [v_pa, v_a, v_b, v_c]
    kwargs_source = aux_params[0]
    v_pa, v_a, v_b, v_c = fitted_params
    c_x, c_y = kwargs_source[0]["center_x"], kwargs_source[0]["center_y"]
    c = 299792

    if not check_list(x):
        return arctan_2d(v_pa, v_a, v_b, v_c, c_x, c_y, x, y) / c
    else:
        return np.array(
            [
                arctan_2d(v_pa, v_a, v_b, v_c, c_x, c_y, xp, yp) / c
                for xp, yp in zip(x, y)
            ]
        )


def get_sersic_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
    # aux_params: [sm, kwargs_source]
    # fitted_params: [scale]
    sm, kwargs_source = aux_params
    scale = fitted_params[0]

    if not check_list(x):
        return (
            sm._light_model.surface_brightness(
                np.array([x]), np.array([y]), kwargs_source
            )[0]
            * scale
        )
    else:
        return sm._light_model.surface_brightness(x, y, kwargs_source) * scale


def get_gaussian_v_given_xy_bin(x, y, binno, aux_params, fitted_params):
    # aux_params: [kwargs_source, sigma_intrinsic] or
    # fitted_params: [amp, sigma_model]
    if not check_list(x):
        kwargs_source, sigma_intrinsic = aux_params
        amp, sigma_model = fitted_params

        c_x, c_y = kwargs_source[0]["center_x"], kwargs_source[0]["center_y"]
        dist = ((x - c_x) ** 2 + (y - c_y) ** 2) ** 0.5

        sigma_final = (sigma_intrinsic**2 + sigma_model**2) ** 0.5

        return norm_dist(amp, 0, sigma_final, dist)
    else:
        return np.array(
            [
                get_gaussian_v_given_xy_bin(x0, y0, binno0, aux_params, fitted_params)
                for x0, y0, binno0 in zip(x, y, binno)
            ]
        )


def get_constant_v_given_xy_bin(x, y, binno, aux_params=[], fitted_params=[]):
    # aux_params: [constant_val] or
    # fitted_params: [constant_val]
    if len(fitted_params) == 0:
        const_val = aux_params[0]
    else:
        const_val = fitted_params[0]

    if not check_list(x):
        return const_val
    else:
        return np.ones(len(x)) * const_val

In [ ]:
ra_grid, dec_grid = ifulmodel4.immodel.ImageNumerics.coordinates_evaluate

x_source, y_source = ifulmodel4.sm._lens_model.ray_shooting(
    ra_grid, dec_grid, flatmodel4.init_pso_fit["kwargs_lens"]
)

source_light = []
inds = ifulmodel4.given_ra_dec_return_bin_no(
    x_source, y_source, flatmodel4.init_pso_fit["kwargs_source"][0]
)

init_fit = copy.deepcopy(imset4.init_spec_fit)
imset4.gen_2d_spec(imset4.init_spec_fit)

source_light = np.array(
    [
        np.zeros(imset4.wavelength.shape)
        if np.isnan(ind)
        else imset4.gen_2d_spec(
            [init_fit[0], init_fit[1], ifulmodel4.init_bin_sourceflux[int(ind)] * 1000]
            + list(init_fit[3:])
        )
        for ind in inds
    ]
)

model_datacube = []
for ii in np.arange(source_light.shape[1]):
    model_datacube += [
        ifulmodel4.immodel.ImageNumerics.re_size_convolve(
            source_light[:, ii], unconvolved=False
        )
    ]
model_datacube = np.array(model_datacube)

In [ ]:
cropped_hdul = fits.HDUList(
    [
        fits.PrimaryHDU(
            data=model_datacube * np.transpose(imset4.mask_3d, (2, 0, 1)),
            header=flatmodel4.header_wcs,
        )
    ]
)
cropped_hdul.writeto("datacubes/test_datacube.fits", overwrite=True)

In [ ]:
mask_3d = np.transpose(imset4.mask_3d, (2, 0, 1))
datacube = np.transpose(imset4.datacube, (2, 0, 1))
datacube.shape

In [ ]:
def save_datacubes(
    header,
    suffix,
    minfunc,
    x0,
    mask_3d,
    var_datacube,
    data_datacube,
    save_data=False,
    show_gif=True,
):
    if "null" in suffix:
        model_datacube = np.zeros(data_datacube.shape)
    else:
        model_datacube = minfunc([x0], True)

    if save_data:
        cropped_hdul = fits.HDUList(
            [fits.PrimaryHDU(data=data_datacube * mask_3d, header=header)]
        )
        cropped_hdul.writeto("datacubes/test_datacube.fits", overwrite=True)

    cropped_hdul = fits.HDUList(
        [fits.PrimaryHDU(data=model_datacube * mask_3d, header=header)]
    )
    cropped_hdul.writeto(f"datacubes/model_datacube{suffix}.fits", overwrite=True)
    cropped_hdul = fits.HDUList(
        [
            fits.PrimaryHDU(
                data=((data_datacube - model_datacube) * mask_3d / var_datacube**0.5)
                * mask_3d,
                header=header,
            )
        ]
    )
    cropped_hdul.writeto(
        f"datacubes/norm_redisuals_datacube{suffix}.fits", overwrite=True
    )
    gen_gif(
        data_datacube,
        model_datacube,
        var_datacube,
        mask_3d,
        trunc_wave_f,
        f"datacubes/animation{suffix}.gif",
    )

    if show_gif:
        return imp(filename=f"datacubes/animation{suffix}.gif")

In [ ]:
imset4.mask_3d.shape

In [ ]:
imset4.datacube.shape

In [ ]:
np.transpose(model_datacube, (1, 2, 0)).shape

In [ ]:
np.transpose(imset4.mask_3d, (2, 0, 1)).shape

In [ ]:
plt.imshow(
    np.log10((model_datacube * np.transpose(imset4.mask_3d, (2, 0, 1)))[15, :, :])
)

In [ ]:
init_fit

In [ ]:
[init_fit[0], init_fit[1], ifulmodel4.init_bin_sourceflux[10] * 1000] + init_fit[3:]

In [ ]:
init_fit[3:]

In [ ]:
plt.plot(imset4.gen_2d_spec([init_fit[0], init_fit[1], 10] + list(init_fit[3:])))